<a href="https://colab.research.google.com/github/poojakrishnas2005/poojakrishnas2005/blob/main/Group_DNA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import numpy as np
from datetime import datetime, timedelta
FMT = '%d/%m/%y, %H:%M'
MEDIA = '<Media omitted>'
DELETED = ('This message was deleted', 'You deleted this message')
STOP = set("i is the a an and or to of in on for it you me my we u ur are was be this that so but with at as if do not no yes he she they them his her its have has had will can just from by am how about which up one hai tha hain ho ka ki ke ko se ne mein bhi toh today entire started telling anyone everyone".split())
PUNCT = '.,!?;:"()[]{}<>*_~`|\\/-+=#@$%^&\u2019\u2018\u201c\u201d'
CARE = ['okay', 'safe', 'eat', 'sleep', 'take care', 'are you', 'please', 'reminder', 'drink water', "don't forget"]
LAUGH = ['lol', 'lmao', 'haha', 'rofl', 'lmfao']
MIN_SAMPLE = 30
ARCH = ['THE SPAMMER', 'THE GROUP MOM', 'THE NIGHT OWL', 'THE STORYTELLER',
        'THE DRAMA QUEEN', 'THE GHOST', 'THE COMEDIAN', 'THE QUESTION MASTER']
THRESH = {'THE SPAMMER': 3.0, 'THE GROUP MOM': 25.0, 'THE NIGHT OWL': 60.0, 'THE STORYTELLER': 30.0,
          'THE DRAMA QUEEN': 30.0, 'THE GHOST': 60.0, 'THE COMEDIAN': 10.0, 'THE QUESTION MASTER': 25.0}

def is_ts_line(line):
    return (len(line) > 17 and line[2] == '/' and line[5] == '/' and
            line[:2].isdigit() and line[3:5].isdigit() and line[6:8].isdigit())

def parse_chat(path):
    with open(path, 'r', encoding='utf-8') as f:
        lines = f.read().split('\n')
    msgs, cur = [], None
    st = {'system': 0, 'media': 0, 'deleted': 0, 'blank': 0, 'malformed': 0, 'continuation': 0}
    for raw in lines:
        line = raw.replace('\u200e', '').rstrip('\r')
        if not line.strip():
            st['blank'] += 1; continue
        if not is_ts_line(line):
            if cur is not None:
                cur['text'] += ' ' + line.strip(); st['continuation'] += 1
            else:
                st['malformed'] += 1
            continue
        if cur is not None: msgs.append(cur); cur = None
        parts = line.split(' - ', 1)
        if len(parts) < 2: st['malformed'] += 1; continue
        try: dt = datetime.strptime(parts[0], FMT)
        except ValueError: st['malformed'] += 1; continue
        if ': ' not in parts[1]:
            st['system'] += 1
            k = parts[1].find('created group "')
            if k != -1 and 'group' not in st: st['group'] = parts[1][k + 15:].rstrip().rstrip('"')
            continue
        sender, text = parts[1].split(': ', 1)
        text = text.strip(); kind = 'text'
        if text == MEDIA: kind = 'media'; st['media'] += 1
        elif text in DELETED: kind = 'deleted'; st['deleted'] += 1
        cur = {'dt': dt, 'sender': sender.strip(), 'text': text, 'kind': kind}
    if cur is not None: msgs.append(cur)
    return msgs, st

def tokenize(text):
    out = []
    for w in text.lower().split():
        w = w.strip(PUNCT)
        if w and w not in STOP and not w.isdigit(): out.append(w)
    return out

def bar(v, vmax, width=20):
    return '█' * (max(1, round(width * v / vmax)) if v > 0 else 0)

def analyse(path):
    msgs, st = parse_chat(path)
    if not msgs: return None
    cnt = {}
    for m in msgs: cnt[m['sender']] = cnt.get(m['sender'], 0) + 1
    people = sorted(cnt, key=lambda p: (-cnt[p], p)); idx = {p: i for i, p in enumerate(people)}
    d0, d1 = msgs[0]['dt'].date(), msgs[-1]['dt'].date(); nd = (d1 - d0).days + 1
    A = {'msgs': msgs, 'st': st, 'cnt': cnt, 'people': people, 'd0': d0, 'd1': d1, 'nd': nd}
    day = {}
    for m in msgs: k = m['dt'].date(); day[k] = day.get(k, 0) + 1
    A['busy_day'] = max(day, key=lambda k: (day[k], -k.toordinal())); A['day'] = day
    H = np.zeros((len(people), 24), dtype=int)
    for m in msgs: H[idx[m['sender']], m['dt'].hour] += 1
    A['H'] = H; A['busy_hour'] = int(H.sum(axis=0).argmax())
    wc = {}; pt = {p: [] for p in people}
    for m in msgs:
        if m['kind'] == 'text':
            pt[m['sender']].append(m['text'])
            for w in tokenize(m['text']): wc[w] = wc.get(w, 0) + 1
    A['wc'] = wc; A['pt'] = pt
    gaps = {p: [] for p in people}
    for i in range(1, len(msgs)):
        if msgs[i-1]['sender'] != msgs[i]['sender']:
            gaps[msgs[i]['sender']].append((msgs[i]['dt'] - msgs[i-1]['dt']).total_seconds())
    A['gaps'] = gaps
    act = {p: set() for p in people}
    for m in msgs: act[m['sender']].add(m['dt'].date())
    A['active'] = act; A['streak'] = {}
    for p in people:
        best = cur = 0; cs = bs = be = None
        for i in range(nd):
            d = d0 + timedelta(days=i)
            if d in act[p]: cur = 0
            else:
                if cur == 0: cs = d
                cur += 1
                if cur > best: best, bs, be = cur, cs, d
        A['streak'][p] = (best, bs, be)
    runs = {p: [] for p in people}; i = 0
    while i < len(msgs):
        j = i
        while j + 1 < len(msgs) and msgs[j+1]['sender'] == msgs[i]['sender']: j += 1
        runs[msgs[i]['sender']].append(j - i + 1); i = j + 1
    A['burst'] = {p: sum(runs[p]) / len(runs[p]) for p in people}

    M = {}
    for p in people:
        t = pt[p]; n = len(t); R = {'n': n}
        R['SPAMMER'] = A['burst'][p]
        R['MOM_pct'] = 100 * sum(1 for x in t if any(k in x.lower() for k in CARE)) / n if n else 0
        R['MOM_hits'] = sum(x.lower().count(k) for x in t for k in CARE)
        night = int(H[idx[p], 23] + H[idx[p], 0:5].sum()); R['OWL'] = 100 * night / int(H[idx[p]].sum())
        R['STORY'] = sum(len(x.split()) for x in t) / n if n else 0
        R['DRAMA'] = 100 * sum(1 for x in t if (len(x) >= 3 and x.isupper()) or x.count('!') >= 2) / n if n else 0
        R['GHOST'] = 100 * (nd - len(act[p])) / nd
        R['LAUGH'] = 100 * sum(1 for x in t if any(k in x.lower() for k in LAUGH)) / n if n else 0
        R['Q'] = 100 * sum(1 for x in t if x.endswith('?')) / n if n else 0
        M[p] = R
    A['M'] = M
    key = {'THE SPAMMER': 'SPAMMER', 'THE GROUP MOM': 'MOM_pct', 'THE NIGHT OWL': 'OWL', 'THE STORYTELLER': 'STORY',
           'THE DRAMA QUEEN': 'DRAMA', 'THE GHOST': 'GHOST', 'THE COMEDIAN': 'LAUGH', 'THE QUESTION MASTER': 'Q'}
    S = {p: {} for p in people}
    for p in people:
        for a in ARCH:
            s = M[p][key[a]] / THRESH[a]
            if a != 'THE GHOST' and A['cnt'][p] < MIN_SAMPLE: s = 0.0
            S[p][a] = s
    A['S'] = S
    pairs = sorted(((S[p][a], p, a) for p in people for a in ARCH if S[p][a] >= 1.0), reverse=True)
    assigned, used = {}, set()
    for s, p, a in pairs:
        if p not in assigned and a not in used: assigned[p] = a; used.add(a)
    for p in people:
        if p not in assigned:
            rest = sorted(((S[p][a], a) for a in ARCH if a not in used), reverse=True)
            assigned[p] = rest[0][1] if rest and rest[0][0] > 0 else 'UNCLASSIFIED'
            if assigned[p] != 'UNCLASSIFIED': used.add(assigned[p])
    A['assign'] = assigned
    return A

def evidence(A, p, a):
    M = A['M'][p]
    return {'THE SPAMMER': f"avg {M['SPAMMER']:.1f} msgs in a row",
            'THE GROUP MOM': f"caring-keyword score: {M['MOM_hits']}",
            'THE NIGHT OWL': f"{M['OWL']:.1f}% msgs between 23h-04h",
            'THE STORYTELLER': f"avg {M['STORY']:.1f} words per msg",
            'THE DRAMA QUEEN': f"{M['DRAMA']:.1f}% ALL-CAPS/!! messages",
            'THE GHOST': f"silent on {A['nd'] - len(A['active'][p])} of {A['nd']} days",
            'THE COMEDIAN': f"{M['LAUGH']:.1f}% laugh-token messages",
            'THE QUESTION MASTER': f"{M['Q']:.1f}% msgs end with '?'",
            'UNCLASSIFIED': 'insufficient data or no threshold met'}[a]

def fmtd(d): return d.strftime('%d %B %Y')

def dashboard(A, name='Hostel Bois 4ever'):
    W = 60; L = []
    P = A['people']; tot = len(A['msgs']); H = A['H']; st = A['st']
    L += ['=' * W, f'  GROUPDNA REPORT — "{name}"', f"  {A['nd']} days • {tot:,} messages • {len(P)} members", '=' * W, '']
    L += [f"  Period              : {fmtd(A['d0'])} to {fmtd(A['d1'])}",
          f"  Busiest day         : {fmtd(A['busy_day'])} ({A['day'][A['busy_day']]} messages)",
          f"  Busiest hour        : {A['busy_hour']:02d}:00 - {(A['busy_hour']+1) % 24:02d}:00 "
          f"(avg {H[:, A['busy_hour']].sum() / A['nd']:.1f} messages per day)",
          f"  Data quality        : {st['system']} system • {st['media']} media • {st['deleted']} deleted", '']
    L.append('  MESSAGES PER PERSON')
    mx = max(A['cnt'].values())
    for p in P:
        L.append(f"    {p:<8}{bar(A['cnt'][p], mx):<21}{A['cnt'][p]:>5} ({100*A['cnt'][p]/tot:4.1f}%)")
    L += ['', '  ACTIVITY HEATMAP (hour of day)', '              ' + ''.join(f'{h:02d} ' if h % 3 == 0 else '   ' for h in range(24))]
    for i, p in enumerate(P):
        row = H[i]; m = row.max(); cells = ''
        for v in row:
            r = v / m if m else 0
            cells += ('.  ' if r < .25 else '░  ' if r < .5 else '▒  ' if r < .75 else '█  ')
        tag = '  <- NIGHT OWL' if A['assign'][p] == 'THE NIGHT OWL' else ''
        L.append(f'    {p:<8}  {cells}{tag}')
    L += ['', "  THIS GROUP'S FAVOURITE WORDS"]
    top = sorted(A['wc'].items(), key=lambda x: (-x[1], x[0]))[:5]
    for w, c in top: L.append(f'    {w:<10}{bar(c, top[0][1]):<21}{c:>5}')
    L += ['', '  RESPONSE PATTERNS']
    avg = {p: sum(g) / len(g) for p, g in A['gaps'].items() if g}
    med = {}
    for p, g in A['gaps'].items():
        if g: s = sorted(g); n = len(s); med[p] = s[n // 2] if n % 2 else (s[n//2 - 1] + s[n//2]) / 2
    def f(sec): return f'{sec/60:.1f} minutes' if sec < 3600 else f'{sec/3600:.1f} hours'
    if avg:
        fp = min(avg, key=avg.get); sp = max(avg, key=avg.get)
        L.append(f'    Fastest replier : {fp:<8} (avg {f(avg[fp])}, median {f(med[fp])})')
        L.append(f'    Slowest replier : {sp:<8} (avg {f(avg[sp])}, median {f(med[sp])})')
    if not avg: L.append('    Not available (needs at least two participants)')
    L += ['', '  LONGEST SILENT STREAKS (consecutive days with zero messages)']
    for p in sorted(P, key=lambda p: -A['streak'][p][0]):
        n, s, e = A['streak'][p]
        if n == 0: L.append(f'    {p:<8}: 0 days (never went silent)')
        else: L.append(f"    {p:<8}: {n} day{'s' if n != 1 else ''} ({s.strftime('%d %b')} - {e.strftime('%d %b')})")
    L += ['', '  PERSONALITY ARCHETYPES']
    for p in P:
        a = A['assign'][p]; L.append(f'    {p:<8}→ {a:<20}({evidence(A, p, a)})')
    L += ['', '=' * W, '  Generated by GroupDNA • Built with Python + NumPy', '=' * W]
    return '\n'.join(L)

file_name = 'hostel_bois.txt'

if os.path.exists(file_name):
    A = analyse(file_name)
    group_title = A['st'].get('group', 'Hostel Bois 4ever')
    report_text = dashboard(A, group_title)
    print(report_text)

    with open('groupdna_report.txt', 'w', encoding='utf-8') as f:
        f.write(report_text)
else:
    print(f"File '{file_name}' not found. Please upload it to the Colab files sidebar.")

  GROUPDNA REPORT — "Hostel Bois 4ever"
  60 days • 3,174 messages • 6 members

  Period              : 01 April 2024 to 30 May 2024
  Busiest day         : 04 May 2024 (76 messages)
  Busiest hour        : 18:00 - 19:00 (avg 4.1 messages per day)
  Data quality        : 4 system • 32 media • 15 deleted

  MESSAGES PER PERSON
    Rahul   ████████████████████   953 (30.0%)
    Priya   ███████████████        718 (22.6%)
    Neha    █████████████          635 (20.0%)
    Aman    ██████████             490 (15.4%)
    Karan   ███████                354 (11.2%)
    Vikas   █                       24 ( 0.8%)

  ACTIVITY HEATMAP (hour of day)
              00       03       06       09       12       15       18       21       
    Rahul     .  .  .  .  .  .  .  .  .  .  .  .  ▒  ░  ░  ▒  ▒  ░  █  ▒  ░  █  ▒  ▒  
    Priya     .  .  .  .  .  .  .  ░  ▒  █  █  █  █  ▒  ▒  ░  ░  ▒  ▒  █  ▒  ░  ░  .  
    Neha      .  .  .  .  .  ░  .  .  ▒  █  █  ░  ▒  ▒  ░  .  ▒  █  █  █  ▒  ░  ░  ░  
    Aman